# WeatherNext 3 → NetCDF for a bounding box (Colab)

Runs [`weathernext3_forecast.py`](https://github.com/<your-github-username>/weathernext3-forecast)
in Colab: 14 days of hourly precipitation and 2 m temperature for a bounding box, written as a
CF-style NetCDF. Earth Engine only — no Cloud Storage bucket and no billing account.

In [ ]:
#@title 1. Get the code and install dependencies
!git clone -q https://github.com/majidniazkar/weathernext3-forecast.git
%cd weathernext3-forecast
!pip install -q -r requirements.txt

In [ ]:
#@title 2. Authenticate Earth Engine (opens a consent page)
import ee

EE_PROJECT = "my-ee-project"  #@param {type:"string"}
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready")

In [ ]:
#@title 3. Settings
WEST, SOUTH, EAST, NORTH = 6.0, 44.0, 14.0, 48.0  # bounding box, degrees
DAYS  = 14      #@param {type:"number"}   # synoptic runs reach 15 days
RES   = "0p1"   #@param ["0p1", "0p05"]   # 0p1 has precipitation; 0p05 is station-head temp/dewpoint
STATS = "mean p10 p90"  #@param {type:"string"}

In [ ]:
#@title 4. Download the forecast
!python weathernext3_forecast.py --project "$EE_PROJECT" --bbox $WEST $SOUTH $EAST $NORTH --days $DAYS --res $RES --stats $STATS

In [ ]:
#@title 5. Inspect and download the file
import glob

import matplotlib.pyplot as plt
import xarray as xr
from google.colab import files

path = sorted(glob.glob("weathernext3_*.nc"))[-1]
ds = xr.open_dataset(path)
print(path, dict(ds.sizes))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ds["total_precipitation_1hr_mean"].sum("time").plot(ax=axes[0], cmap="BuPu")
axes[0].set_title(f"Total precipitation, {int(ds.lead_hours.max())} h")
ds["temperature_2m_mean"].mean(("latitude", "longitude")).plot(ax=axes[1])
axes[1].set_title("Area-mean 2 m temperature")
fig.tight_layout()

files.download(path)

Notes: only the 00/06/12/18 UTC runs extend to 15 days (interim hourly runs stop at 48 h — the
script picks a synoptic run by default), Earth Engine serves ensemble statistics rather than the
64 members, and precipitation is native 0.1°: there is no 5 km precipitation field.